In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Confirm we're on Kaggle CPU with internet.
import sys, platform, os
print("Python:", sys.version.split()[0], "on", platform.system())
print("Working dir:", os.getcwd())
# Quick internet probe
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=5).status
    print("Internet: OK (huggingface.co reachable)")
except Exception as e:
    raise RuntimeError(
        f"Internet unreachable: {e}\n"
        "Fix: Notebook settings (right sidebar) → Internet → ON, then Save & rerun."
    )

Python: 3.12.13 on Linux
Working dir: /kaggle/working
Internet: OK (huggingface.co reachable)


In [3]:
# MTEB 2.x pulls in the right sentence-transformers + torch. We uninstall
# torchvision because MTEB's optional imports trigger a torchvision::nms
# error on Kaggle's default torch/torchvision mismatch. We don't need it.
!pip uninstall -y torchvision torchaudio -q
!pip install -q "mteb==2.21.0" "sentence-transformers>=3.0" rank-bm25
print("done — dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 10.4 MB/s eta 0:00:00
done — dependencies installed.


In [4]:
import subprocess, pathlib, shutil

REPO_DIR = "/kaggle/working/samsung-prism"
REPO_URL = "https://github.com/Ajha2005/samsung-prism.git"

# Fresh clone every time so we always get the latest main.
if pathlib.Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
print("Cloned at:", REPO_DIR)
print("Head:", subprocess.check_output(["git", "-C", REPO_DIR, "log", "-1", "--oneline"]).decode().strip())

# Sanity: confirm the two new configs are there.
for cfg in ["config/unixcoder.json", "config/mpnet.json", "config/baseline.json"]:
    p = pathlib.Path(REPO_DIR) / cfg
    print(f"  {cfg}: {'OK' if p.exists() else 'MISSING'}")

Cloning into '/kaggle/working/samsung-prism'...


Cloned at: /kaggle/working/samsung-prism
Head: 4275ccd Add two candidate configs for the last high-EV shot at the score
  config/unixcoder.json: OK
  config/mpnet.json: OK
  config/baseline.json: OK


In [5]:
# Run the offline hashing-backend demo. It uses zero network and zero
# HuggingFace, so if this fails the problem is dependency-side, not model-side.
import subprocess
r = subprocess.run(
    ["python", "-m", "prism.cli", "demo", "--backend", "hashing"],
    cwd=REPO_DIR, capture_output=True, text=True,
)
print(r.stdout[-1500:])          # last 1500 chars are the ranked results
if r.returncode != 0:
    print("STDERR:", r.stderr[-800:])
    raise RuntimeError("Smoke test failed — do NOT run the eval cell until this passes.")
print("\n✅ smoke test passed — safe to run the eval cell.")

[prism-demo] indexing 18 snippets | config: demo [model=all-MiniLM-L6-v2]
[prism-demo] index built in 12.2 ms

Q: find two numbers that sum to a target   (9.3 ms)
  1. [two_sum]  score=0.3810   def two_sum(nums, target):
  2. [binary_search]  score=0.1483   def binary_search(arr, target):
  3. [lru_cache_manual]  score=0.0785   from collections import OrderedDict

Q: detect if a string is a palindrome   (0.5 ms)
  1. [is_palindrome]  score=0.4527   def is_palindrome(s):
  2. [gcd]  score=0.1690   def gcd(a, b):
  3. [fibonacci]  score=0.1151   def fib(n):

Q: shortest path in an unweighted graph   (0.3 ms)
  1. [bfs]  score=0.1548   from collections import deque
  2. [anagram_groups]  score=0.1360   from collections import defaultdict
  3. [dfs]  score=0.1330   def dfs(graph, start):

Q: remove duplicate words and count them   (0.3 ms)
  1. [word_count]  score=0.2140   from collections import Counter
  2. [anagram_groups]  score=0.1120   from collections import defaultdict
  3. [lru_ca

In [6]:
# List everything you can download. In the Kaggle notebook UI, files under
# /kaggle/working are downloadable from the right sidebar → Output tab.
import pathlib
for p in sorted(pathlib.Path("/kaggle/working/results").glob("*.json")):
    print(f"{p.name:<45} {p.stat().st_size:>10} bytes")

print()
print("NEXT STEPS")
print("=" * 72)
print("1. If unixcoder or mpnet BEAT 0.0662:")
print("   - Download the winning *_appsretrieval_results.json from the sidebar")
print("   - Rename locally to  appsretrieval_results.json")
print("   - Attach that file to your GitHub release  (replaces the baseline JSON)")
print("   - Tell me the number and I'll rebuild slide 8 + ablation log")
print()
print("2. If BOTH lose:")
print("   - Baseline stays submitted (nothing to change on the release)")
print("   - Tell me both numbers — I'll add them to docs/ablation_log.md")
print("     as experiments 7 and 8, making the discipline story stronger.")


NEXT STEPS
1. If unixcoder or mpnet BEAT 0.0662:
   - Download the winning *_appsretrieval_results.json from the sidebar
   - Rename locally to  appsretrieval_results.json
   - Attach that file to your GitHub release  (replaces the baseline JSON)
   - Tell me the number and I'll rebuild slide 8 + ablation log

2. If BOTH lose:
   - Baseline stays submitted (nothing to change on the release)
   - Tell me both numbers — I'll add them to docs/ablation_log.md
     as experiments 7 and 8, making the discipline story stronger.


In [7]:
# Reinstall mteb LOUDLY so any failure is visible, then verify.
import subprocess, sys

print("=" * 70)
print("Re-installing mteb (verbose this time)")
print("=" * 70)
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade",
     "mteb==2.21.0", "sentence-transformers>=3.0", "rank-bm25"],
    capture_output=True, text=True,
)
print(r.stdout[-3000:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-2000:])
    raise RuntimeError("pip install failed — read the errors above and paste them to me.")

# Verify the imports the eval will need.
print("\n" + "=" * 70)
print("Verifying imports")
print("=" * 70)
for name in ("mteb", "sentence_transformers", "torch"):
    try:
        mod = __import__(name)
        print(f"  ✓ {name}  version={getattr(mod, '__version__', '?')}")
    except Exception as e:
        print(f"  ✗ {name}  FAILED: {e}")
        raise

print("\n✅ Ready. Re-run cell 5 now — it should actually work this time.")

Re-installing mteb (verbose this time)
3.0.3)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 9.5 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.4.1
    Uninstalling sentence-transformers-5.4.1:
      Successfully uninstalled sentence-transformers-5.4.1


Verifying imports
  ✓ mteb  version=2.21.0
  ✓ sentence_transformers  version=6.1.0
  ✓ torch  version=2.10.0+cpu

✅ Ready. Re-run cell 5 now — it should actually work this time.


In [8]:
import subprocess, pathlib, json, time

OUT = pathlib.Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)

BASELINE_NDCG = 0.0662  # the number we're trying to beat

# Highest-EV first. If Kaggle's 12-hr window looks tight, unixcoder alone
# is still worth it — it aligns with the earlier ablation's diagnosis.
CANDIDATES = [
    ("unixcoder", "config/unixcoder.json"),
    ("mpnet",     "config/mpnet.json"),
]

results = {}
for name, cfg in CANDIDATES:
    out_path = OUT / f"{name}_appsretrieval_results.json"
    print()
    print("=" * 72)
    print(f"▶ {name}   config={cfg}   output={out_path.name}")
    print("=" * 72)
    t0 = time.time()

    r = subprocess.run(
        ["python", "-m", "prism.cli", "eval",
         "--config", cfg,
         "--output", str(out_path)],
        cwd=REPO_DIR,
    )
    elapsed = time.time() - t0
    print(f"[{name}] exit={r.returncode}  elapsed={elapsed/60:.1f} min")

    if r.returncode != 0 or not out_path.exists():
        print(f"[{name}] FAILED — moving on to next.")
        results[name] = None
        continue

    d = json.loads(out_path.read_text())
    ndcg = d["scores"]["test"][0]["ndcg_at_10"]
    mrr  = d["scores"]["test"][0].get("mrr_at_10", None)
    delta = (ndcg - BASELINE_NDCG) / BASELINE_NDCG * 100
    verdict = "🏆 WIN — swap into submission" if ndcg > BASELINE_NDCG else "↩︎  drop — baseline stays submitted"
    results[name] = {"ndcg": ndcg, "mrr": mrr, "delta_pct": delta, "verdict": verdict}
    print(f"[{name}] NDCG@10={ndcg:.4f}  MRR@10={mrr}  ({delta:+.1f}% vs baseline)  {verdict}")

# Final scoreboard
print()
print("=" * 72)
print("SCOREBOARD  ·  baseline NDCG@10 = 0.0662")
print("=" * 72)
print(f"{'config':<12} {'NDCG@10':>10} {'MRR@10':>10} {'Δ vs base':>12}  verdict")
print("-" * 72)
for name, res in results.items():
    if res is None:
        print(f"{name:<12} {'FAILED':>10}")
    else:
        mrr = f"{res['mrr']:.4f}" if res['mrr'] is not None else "—"
        print(f"{name:<12} {res['ndcg']:>10.4f} {mrr:>10} {res['delta_pct']:>+11.1f}%  {res['verdict']}")


▶ unixcoder   config=config/unixcoder.json   output=unixcoder_appsretrieval_results.json


[prism-eval] config: unixcoder-experiment [model=unixcoder-base]
[prism-eval] running AppsRetrieval (downloads model + dataset on first run)...
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2471.15it/s, Materializing param=pooler.dense.weight]
RobertaModel LOAD REPORT from: microsoft/unixcoder-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/kaggle/working/samsung-prism/prism/backends/sentence_transformer.py:50: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = self.model.get_sentence_embedding_dimension()
Standardizing text corpus format: 100%|██████████| 8765/8765 [00:00<00:00, 11397.28 examples/s]


[prism-eval] NDCG@10 = 0.04337
[prism-eval] MRR@10  = 0.03480796391154957
[prism-eval] wrote /kaggle/working/results/unixcoder_appsretrieval_results.json  <- upload THIS one (raw task_result.to_dict(), matches the problem statement's own reference snippet exactly)
[prism-eval] wrote /kaggle/working/results/unixcoder_appsretrieval_results.debug.json  (local convenience only, not for submission)
[unixcoder] exit=0  elapsed=101.4 min
[unixcoder] NDCG@10=0.0434  MRR@10=0.03480796391154957  (-34.5% vs baseline)  ↩︎  drop — baseline stays submitted

▶ mpnet   config=config/mpnet.json   output=mpnet_appsretrieval_results.json


[prism-eval] config: mpnet-general-experiment [model=all-mpnet-base-v2]
[prism-eval] running AppsRetrieval (downloads model + dataset on first run)...
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1298.01it/s, Materializing param=pooler.dense.weight]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/kaggle/working/samsung-prism/prism/backends/sentence_transformer.py:50: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = self.model.get_sentence_embedding_dimension()


[prism-eval] NDCG@10 = 0.08371
[prism-eval] MRR@10  = 0.071386517422374
[prism-eval] wrote /kaggle/working/results/mpnet_appsretrieval_results.json  <- upload THIS one (raw task_result.to_dict(), matches the problem statement's own reference snippet exactly)
[prism-eval] wrote /kaggle/working/results/mpnet_appsretrieval_results.debug.json  (local convenience only, not for submission)
[mpnet] exit=0  elapsed=112.9 min
[mpnet] NDCG@10=0.0837  MRR@10=0.071386517422374  (+26.5% vs baseline)  🏆 WIN — swap into submission

SCOREBOARD  ·  baseline NDCG@10 = 0.0662
config          NDCG@10     MRR@10    Δ vs base  verdict
------------------------------------------------------------------------
unixcoder        0.0434     0.0348       -34.5%  ↩︎  drop — baseline stays submitted
mpnet            0.0837     0.0714       +26.5%  🏆 WIN — swap into submission


In [9]:
# Check what's on disk and any in-flight download.
import subprocess, pathlib
print("Results so far:")
for p in sorted(pathlib.Path("/kaggle/working/results").glob("*.json")):
    print(f"  {p.name:<45} {p.stat().st_size} bytes")

print("\nHF cache (model download progress):")
for p in pathlib.Path("/root/.cache/huggingface/hub").glob("**/pytorch_model.bin"):
    print(f"  {p.parent.parent.name}: {p.stat().st_size/1e6:.1f} MB")
for p in pathlib.Path("/root/.cache/huggingface/hub").glob("**/model.safetensors"):
    print(f"  {p.parent.parent.name}: {p.stat().st_size/1e6:.1f} MB")

print("\nRunning python processes:")
print(subprocess.check_output(["ps", "-ef"]).decode().split("\n")[0])
for line in subprocess.check_output(["ps", "-ef"]).decode().splitlines():
    if "prism.cli" in line or "python" in line and "prism" in line:
        print(line)

Results so far:
  mpnet_appsretrieval_results.debug.json        8147 bytes
  mpnet_appsretrieval_results.json              8204 bytes
  unixcoder_appsretrieval_results.debug.json    8126 bytes
  unixcoder_appsretrieval_results.json          8198 bytes

HF cache (model download progress):
  snapshots: 503.8 MB
  snapshots: 438.0 MB
  .no_exist: 0.0 MB
  snapshots: 503.7 MB

Running python processes:
UID          PID    PPID  C STIME TTY          TIME CMD


In [10]:
import json
d = json.load(open("/kaggle/working/results/unixcoder_appsretrieval_results.json"))
print("unixcoder NDCG@10 =", d["scores"]["test"][0]["ndcg_at_10"])

unixcoder NDCG@10 = 0.04337
